In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

project_path = Path.cwd()

if project_path.name == "notebooks":
    project_path = project_path.parent

data_path = project_path / "data" / "processed"
reports_path = project_path / "docs"
reports_path.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(data_path / "train.csv")
val_df = pd.read_csv(data_path / "validation.csv")
test_df = pd.read_csv(data_path / "test.csv")
aug_df = pd.read_csv(data_path / "augmentation_candidates.csv")

print("Original Training:", len(train_df))
print("Validation:", len(val_df))
print("Testing:", len(test_df))
print("Augmentation Candidates:", len(aug_df))

print("\nAugmentation columns:")
print(aug_df.columns.tolist())

display(aug_df.head(3))

Original Training: 803
Validation: 178
Testing: 172
Augmentation Candidates: 92

Augmentation columns:
['label', 'original_text', 'text', 'group_id', 'method']


,label,original_text,text,group_id,method
0,Psoriasis,I have been experiencing a skin rash on my arm...,"I have a skin rash on my arms, legs, and torso...",0,controlled_replacement
1,Psoriasis,I have been experiencing joint pain in my fing...,"I have joint pain in my fingers, wrists, and k...",2,controlled_replacement
2,Psoriasis,"My skin is prone to infections due to dry, fla...","My skin is prone to infections due to dry, fla...",14,controlled_replacement


In [2]:
required_columns = {
    "label", "original_text", "text", "group_id"
}

missing_columns = required_columns - set(aug_df.columns)

if missing_columns:
    raise ValueError(
        f"Missing columns: {missing_columns}"
    )

missing_values = aug_df[
    ["label", "original_text", "text", "group_id"]
].isna().sum()

duplicate_texts = aug_df["text"].duplicated().sum()

unchanged_texts = (
    aug_df["original_text"] == aug_df["text"]
).sum()

invalid_labels = (
    ~aug_df["label"].isin(train_df["label"])
).sum()

empty_texts = (
    aug_df["text"].fillna("").str.strip() == ""
).sum()

print("DATA QUALITY CHECK")
print("------------------")
print("Missing values:")
print(missing_values.to_string())

print("\nDuplicate generated texts:", duplicate_texts)
print("Unchanged generated texts:", unchanged_texts)
print("Invalid disease labels:", invalid_labels)
print("Empty generated texts:", empty_texts)

DATA QUALITY CHECK
------------------
Missing values:
label            0
original_text    0
text             0
group_id         0

Duplicate generated texts: 0
Unchanged generated texts: 0
Invalid disease labels: 0
Empty generated texts: 0


In [3]:
def normalize_text(text):
    return " ".join(
        str(text).lower().split()
    )

aug_texts = aug_df["text"].map(normalize_text)

train_texts = set(
    train_df["text"].map(normalize_text)
)

val_texts = set(
    val_df["text"].map(normalize_text)
)

test_texts = set(
    test_df["text"].map(normalize_text)
)

overlap_train = aug_texts.isin(train_texts).sum()
overlap_val = aug_texts.isin(val_texts).sum()
overlap_test = aug_texts.isin(test_texts).sum()

print("EXACT TEXT OVERLAP CHECK")
print("------------------------")
print("Overlap with Training:", overlap_train)
print("Overlap with Validation:", overlap_val)
print("Overlap with Testing:", overlap_test)

EXACT TEXT OVERLAP CHECK
------------------------
Overlap with Training: 0
Overlap with Validation: 0
Overlap with Testing: 0


In [4]:
import re

replacements = [
    (r"\bI have been experiencing\b", "I have"),
    (r"\bI am experiencing\b", "I have"),
    (r"\bI've been experiencing\b", "I have")
]

def expected_augmentation(text):
    result = str(text)

    for pattern, replacement in replacements:
        result = re.sub(
            pattern,
            replacement,
            result,
            flags=re.IGNORECASE
        )

    return result

aug_df["expected_text"] = aug_df[
    "original_text"
].apply(expected_augmentation)

aug_df["rule_match"] = (
    aug_df["expected_text"] == aug_df["text"]
)

print("TRANSFORMATION CHECK")
print("--------------------")
print("Matching defined rules:",
      aug_df["rule_match"].sum())

print("Unexpected transformations:",
      (~aug_df["rule_match"]).sum())

display(
    aug_df.loc[
        ~aug_df["rule_match"],
        ["original_text", "text", "expected_text"]
    ].head(10)
)

TRANSFORMATION CHECK
--------------------
Matching defined rules: 90
Unexpected transformations: 2


,original_text,text,expected_text
67,"My muscles lack strength, and I have been expe...","My muscles lack strength, and I have back pain...","My muscles lack strength, and I have back pain..."
68,"I've been experiencing back pain, a dry cough,...","I have back pain, a dry cough, and muscle weak...","I have back pain, a dry cough, and muscle weak..."


In [5]:
augmentation_distribution = (
    aug_df["label"]
    .value_counts()
    .rename_axis("Disease")
    .reset_index(name="Generated Samples")
)

print("AUGMENTATION DISTRIBUTION")
print("-------------------------")

display(augmentation_distribution)

print("Classes augmented:",
      aug_df["label"].nunique())

print("Classes not augmented:",
      len(set(train_df["label"]) - set(aug_df["label"])))

AUGMENTATION DISTRIBUTION
-------------------------


,Disease,Generated Samples
0,Cervical spondylosis,11
1,Hypertension,10
2,Dengue,8
3,Typhoid,7
4,Malaria,7
5,Arthritis,6
6,Migraine,6
7,Jaundice,6
8,Dimorphic Hemorrhoids,5
9,urinary tract infection,4


Classes augmented: 22
Classes not augmented: 2


In [6]:
review_samples = aug_df[
    ["label", "original_text", "text"]
].sample(
    n=min(15, len(aug_df)),
    random_state=42
)

pd.set_option("display.max_colwidth", 250)

print("MANUAL REVIEW SAMPLES")
print("---------------------")

display(review_samples)

MANUAL REVIEW SAMPLES
---------------------


,label,original_text,text
40,Arthritis,"Lately i have been experiencing muscle weakness, a stiff neck, swelling joints, and stiffness after I attempt to move around. it is also been very painful to steer.","Lately I have muscle weakness, a stiff neck, swelling joints, and stiffness after I attempt to move around. it is also been very painful to steer."
22,Dengue,"I am experiencing very high fever and chills every night. It is really concerning me. Moreover, I don’t feel like eating anything and my back, arms, legs pain a lot. There is a strange pain behing my eyes. I can’t do any physical activities.","I have very high fever and chills every night. It is really concerning me. Moreover, I don’t feel like eating anything and my back, arms, legs pain a lot. There is a strange pain behing my eyes. I can’t do any physical activities."
55,Migraine,"I have been experiencing acid reflux and difficulty digesting my food, along with regular headaches and impaired vision","I have acid reflux and difficulty digesting my food, along with regular headaches and impaired vision"
72,Jaundice,I have been having severe itching and vomiting. I am also very tired and have lost weight. My fever is high and my skin has turned yellow. My urine is dark and I have been experiencing abdominal pain,I have been having severe itching and vomiting. I am also very tired and have lost weight. My fever is high and my skin has turned yellow. My urine is dark and I have abdominal pain
0,Psoriasis,"I have been experiencing a skin rash on my arms, legs, and torso for the past few weeks. It is red, itchy, and covered in dry, scaly patches.","I have a skin rash on my arms, legs, and torso for the past few weeks. It is red, itchy, and covered in dry, scaly patches."
26,Common Cold,"I've been coughing a lot and feeling chilly and shivery. My nose has been quite clogged, and I am experiencing facial pressure. I also have a lot of phlegm, and coughing pains my chest. I can't smell anything, and my muscles are aching.","I've been coughing a lot and feeling chilly and shivery. My nose has been quite clogged, and I have facial pressure. I also have a lot of phlegm, and coughing pains my chest. I can't smell anything, and my muscles are aching."
39,Arthritis,"I've been experiencing muscular weakness and stiffness in my neck recently. My joints have swollen, creating it troublesome on behalf of me to run regarding while not changing into stiff. Walking has additionally been agonizingly uncomfortable.","I have muscular weakness and stiffness in my neck recently. My joints have swollen, creating it troublesome on behalf of me to run regarding while not changing into stiff. Walking has additionally been agonizingly uncomfortable."
67,Cervical spondylosis,"My muscles lack strength, and I have been experiencing back pain and a dry cough. I've been experiencing neck pain and feeling lightheaded and unstable on my feet.","My muscles lack strength, and I have back pain and a dry cough. I've been experiencing neck pain and feeling lightheaded and unstable on my feet."
10,Typhoid,"I am experiencing a lot of nausea and vomiting, and it's been quite difficult for me to eat anything. I've entirely lost my appetite, and as a result, I have become quite weak.","I have a lot of nausea and vomiting, and it's been quite difficult for me to eat anything. I've entirely lost my appetite, and as a result, I have become quite weak."
44,Hypertension,"I have been experiencing a headache, chest pain, dizziness, and difficulty maintaining my balance. I also feel like I am having trouble concentrating","I have a headache, chest pain, dizziness, and difficulty maintaining my balance. I also feel like I am having trouble concentrating"


In [7]:
report_lines = [
    "BACHELOR THESIS - AUGMENTATION EVALUATION",
    "=" * 50,
    f"Training samples: {len(train_df)}",
    f"Validation samples: {len(val_df)}",
    f"Testing samples: {len(test_df)}",
    f"Augmentation candidates: {len(aug_df)}",
    "",
    "DATA QUALITY",
    f"Missing values: {missing_values.sum()}",
    f"Duplicate generated texts: {duplicate_texts}",
    f"Unchanged texts: {unchanged_texts}",
    f"Invalid labels: {invalid_labels}",
    f"Empty generated texts: {empty_texts}",
    "",
    "EXACT OVERLAP",
    f"Training overlap: {overlap_train}",
    f"Validation overlap: {overlap_val}",
    f"Testing overlap: {overlap_test}",
    "",
    "TRANSFORMATION CHECK",
    f"Matching rules: {aug_df['rule_match'].sum()}",
    f"Unexpected transformations: {(~aug_df['rule_match']).sum()}",
    "",
    "AUGMENTATION DISTRIBUTION",
    augmentation_distribution.to_string(index=False),
    "",
    "MANUAL REVIEW SAMPLES",
    review_samples.to_string(index=False),
]

report_text = "\n".join(map(str, report_lines))

report_file = reports_path / "augmentation_evaluation_report.txt"

report_file.write_text(
    report_text,
    encoding="utf-8"
)

print("Report saved successfully!")
print("Location:", report_file)

Report saved successfully!
Location: c:\Projects\Bachelor-Thesis\docs\augmentation_evaluation_report.txt


In [8]:
from difflib import ndiff

for idx in [67, 68]:
    row = aug_df.loc[idx]

    print("=" * 65)
    print("ROW:", idx)
    print("DISEASE:", row["label"])
    print("\nORIGINAL:")
    print(repr(row["original_text"]))
    print("\nGENERATED:")
    print(repr(row["text"]))
    print("\nEXPECTED:")
    print(repr(row["expected_text"]))

    print("\nDIFFERENCES:")
    print("".join(ndiff(
        row["expected_text"],
        row["text"]
    )))

ROW: 67
DISEASE: Cervical spondylosis

ORIGINAL:
"My muscles lack strength, and I have been experiencing back pain and a dry cough. I've been experiencing neck pain and feeling lightheaded and unstable on my feet."

GENERATED:
"My muscles lack strength, and I have back pain and a dry cough. I've been experiencing neck pain and feeling lightheaded and unstable on my feet."

EXPECTED:
'My muscles lack strength, and I have back pain and a dry cough. I have neck pain and feeling lightheaded and unstable on my feet.'

DIFFERENCES:
  M  y     m  u  s  c  l  e  s     l  a  c  k     s  t  r  e  n  g  t  h  ,     a  n  d     I     h  a  v  e     b  a  c  k     p  a  i  n     a  n  d     a     d  r  y     c  o  u  g  h  .     I+ '-  - h- a  v  e+  + b+ e+ e+ n+  + e+ x+ p+ e+ r+ i+ e+ n+ c+ i+ n+ g     n  e  c  k     p  a  i  n     a  n  d     f  e  e  l  i  n  g     l  i  g  h  t  h  e  a  d  e  d     a  n  d     u  n  s  t  a  b  l  e     o  n     m  y     f  e  e  t  .
ROW: 68
DISEASE: Cervic

In [9]:
import re

# Korrigjojme vetem dy rreshtat
for idx in [67, 68]:
    aug_df.loc[idx, "text"] = expected_augmentation(
        aug_df.loc[idx, "original_text"]
    )

# Kontrollojme perseri transformimet
aug_df["expected_text"] = aug_df[
    "original_text"
].apply(expected_augmentation)

aug_df["rule_match"] = (
    aug_df["text"] == aug_df["expected_text"]
)

print("Matching rules:", aug_df["rule_match"].sum())
print("Unexpected transformations:",
      (~aug_df["rule_match"]).sum())

Matching rules: 92
Unexpected transformations: 0


In [10]:
corrected_path = (
    data_path / "augmentation_candidates_corrected.csv"
)

aug_df[
    ["label", "original_text", "text", "group_id", "method"]
].to_csv(
    corrected_path,
    index=False
)

print("Corrected augmentation candidates saved!")
print("Location:", corrected_path)

Corrected augmentation candidates saved!
Location: c:\Projects\Bachelor-Thesis\data\processed\augmentation_candidates_corrected.csv


In [11]:
import pandas as pd

# Lexojme versionin e korrigjuar
corrected_df = pd.read_csv(
    data_path / "augmentation_candidates_corrected.csv"
)

# Normalizojme tekstet per krahasim
def normalize_text(text):
    return " ".join(str(text).lower().split())

corrected_texts = corrected_df["text"].map(normalize_text)

# Kontrollojme perseritjet
duplicates = corrected_texts.duplicated().sum()

# Kontrollojme mbivendosjet
overlap_train = corrected_texts.isin(
    train_df["text"].map(normalize_text)
).sum()

overlap_val = corrected_texts.isin(
    val_df["text"].map(normalize_text)
).sum()

overlap_test = corrected_texts.isin(
    test_df["text"].map(normalize_text)
).sum()

# Kontrollojme qe origjinalet i perkasin Training
originals_in_train = corrected_df["original_text"].map(
    normalize_text
).isin(set(train_df["text"].map(normalize_text))).sum()

print("FINAL AUGMENTATION CHECK")
print("------------------------")
print("Total candidates:", len(corrected_df))
print("Duplicate generated texts:", duplicates)
print("Overlap with Training:", overlap_train)
print("Overlap with Validation:", overlap_val)
print("Overlap with Testing:", overlap_test)
print("Original texts found in Training:", originals_in_train)

FINAL AUGMENTATION CHECK
------------------------
Total candidates: 92
Duplicate generated texts: 0
Overlap with Training: 0
Overlap with Validation: 0
Overlap with Testing: 0
Original texts found in Training: 92


In [3]:
import pandas as pd
import re
from pathlib import Path

project_path = Path.cwd()

if project_path.name == "notebooks":
    project_path = project_path.parent

data_path = project_path / "data" / "processed"

aug_df = pd.read_csv(
    data_path / "augmentation_candidates_corrected.csv"
)

patterns = [
    r"\bI have been experiencing\b",
    r"\bI am experiencing\b",
    r"\bI've been experiencing\b"
]

def normalize_allowed_phrases(text):
    result = str(text)

    for pattern in patterns:
        result = re.sub(
            pattern, "I have", result,
            flags=re.IGNORECASE
        )

    return result

aug_df["semantic_rule_check"] = aug_df.apply(
    lambda row:
        normalize_allowed_phrases(row["original_text"])
        == row["text"],
    axis=1
)

print("Total candidates:", len(aug_df))
print("Passed rule check:", aug_df["semantic_rule_check"].sum())
print("Failed rule check:", (~aug_df["semantic_rule_check"]).sum())

Total candidates: 92
Passed rule check: 92
Failed rule check: 0


In [4]:
from IPython.display import display

# Shfaqim te 92 variantet per kontroll manual
review_df = aug_df[
    ["label", "original_text", "text"]
].copy()

review_df["Meaning Preserved"] = ""
review_df["Comments"] = ""

display(review_df)

# Ruajme tabelen per kontroll
review_path = (
    project_path / "docs" /
    "augmentation_semantic_review.csv"
)

review_df.to_csv(review_path, index=False)

print("Review file saved:", review_path)
print("Total samples to review:", len(review_df))

,label,original_text,text,Meaning Preserved,Comments
0,Psoriasis,I have been experiencing a skin rash on my arm...,"I have a skin rash on my arms, legs, and torso...",,
1,Psoriasis,I have been experiencing joint pain in my fing...,"I have joint pain in my fingers, wrists, and k...",,
2,Psoriasis,"My skin is prone to infections due to dry, fla...","My skin is prone to infections due to dry, fla...",,
3,Varicose Veins,I have noticed that the blood vessels in my le...,I have noticed that the blood vessels in my le...,,
4,Varicose Veins,I am experiencing too many cramps in the last ...,I have too many cramps in the last couple of d...,,
...,...,...,...,...,...
87,urinary tract infection,"My urine is frequently black, red, and has a r...","My urine is frequently black, red, and has a r...",,
88,allergy,I have been experiencing a loss of appetite an...,I have a loss of appetite and have difficulty ...,,
89,drug reaction,I am experiencing a decrease in my sex drive a...,I have a decrease in my sex drive and difficul...,,
90,drug reaction,I am experiencing changes in my menstrual cycl...,I have changes in my menstrual cycle and unexp...,,


Review file saved: c:\Projects\Bachelor-Thesis\docs\augmentation_semantic_review.csv
Total samples to review: 92
